# Is That Isekai Tower Too High?

In isekai fantasy, the hero often arrives in a medieval-looking town and spots an impossibly tall tower on the horizon. But how tall is *too* tall? This notebook checks whether a tower's height is plausible given how far away you are standing and how far you have to tilt your neck to see the top.

## The Sight-Line Problem

When you stand at horizontal distance $D$ from a tower of height $H$, the elevation angle $\theta$ (neck tilt) to see the top is:

$$\theta = \arctan\left(\frac{H}{D}\right)$$

Rearranging:

$$H = D \cdot \tan(\theta) \qquad D = \frac{H}{\tan(\theta)}$$

If $\theta$ exceeds what a human can comfortably look upward, the tower is "too high" for that viewing distance.

In [ ]:
import math

# --- Human FOV / neck comfort thresholds (degrees above horizontal) ---
VERTICAL_FOV_TOTAL_DEG = 150.0
UPWARD_GAZE_NO_HEAD_MOVEMENT_DEG = 20.0
COMFORTABLE_NECK_TILT_DEG = 15.0
ERGONOMIC_UPPER_BOUND_DEG = 45.0
MAX_UPWARD_HEAD_TILT_DEG = 65.0

# --- Medieval town scale (meters) ---
TYPICAL_BASTIDE_DIAMETER_M = 500.0
LARGE_WALLED_CITY_DIAMETER_M = 1600.0
WALKING_DISTANCE_TOWN_LIMIT_M = 1000.0


def neck_tilt_degrees(distance_m: float, height_m: float) -> float:
    """Elevation angle (degrees) to see the top of a tower."""
    return math.degrees(math.atan(height_m / distance_m))


def height_from_tilt(distance_m: float, tilt_deg: float) -> float:
    """Tower height implied by distance and neck tilt."""
    return distance_m * math.tan(math.radians(tilt_deg))


def distance_from_tilt(height_m: float, tilt_deg: float) -> float:
    """Viewing distance implied by tower height and neck tilt."""
    return height_m / math.tan(math.radians(tilt_deg))


def is_tower_too_high(distance_m: float, height_m: float) -> str:
    """Verdict based on neck tilt vs human FOV comfort thresholds."""
    tilt = neck_tilt_degrees(distance_m, height_m)
    if tilt <= COMFORTABLE_NECK_TILT_DEG:
        return "comfortable"
    if tilt <= ERGONOMIC_UPPER_BOUND_DEG:
        return "uncomfortable"
    if tilt <= MAX_UPWARD_HEAD_TILT_DEG:
        return "strained"
    return "too high"


# Example: 100 m tower at various medieval-scale distances
tower_height_m = 100.0
print(f"{'Distance (m)':>14}  {'Neck tilt (°)':>14}  {'Verdict':>16}")
print("-" * 48)
for distance_m in [200, 500, 1000]:
    tilt = neck_tilt_degrees(distance_m, tower_height_m)
    verdict = is_tower_too_high(distance_m, tower_height_m)
    print(f"{distance_m:>14.0f}  {tilt:>14.1f}  {verdict:>16}")

## Remarks: Human FOV and Medieval Town Scale

### Average Human Field of View

- [**Field of view (Wikipedia)**](https://en.wikipedia.org/wiki/Field_of_view): The vertical visual field in humans is around **150°** total.
- **Upward gaze without head movement**: roughly **20°** above the horizontal line of sight.
- **Comfortable neck tilt** for extended viewing: about **15°** (no subjective fatigue).
- **Ergonomic upper bound**: around **45°** before sustained discomfort.
- **Maximum upward head tilt** (with neck rotation): roughly **65°** — beyond this, seeing the top of a tower becomes physically impractical.

### Average Medieval City / Town Size

- **Typical bastide / fortified town**: defensive perimeter around **500 m** in diameter (~10 hectares).
- **Large walled city**: roughly **0.5–1 sq mi** (~800–1600 m across within the walls).
- **Walking-distance constraint**: most settlements stayed under **~1 km** across because daily life required everything to be within reasonable walking distance.

Standing at a town wall (~250 m from center) or just outside (~500 m), a tower in the town center must stay short enough that your neck tilt stays within the comfort zone — otherwise, that isekai tower really is too high.

## Create and Display the Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        plt.style.use('default')
        print("Using default matplotlib style")

fig, ax = plt.subplots(figsize=(12, 8))

distances_m = np.linspace(50, 2000, 500)
tower_heights_m = [50, 100, 300, 1000]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

for height_m, color in zip(tower_heights_m, colors):
    tilts = np.degrees(np.arctan(height_m / distances_m))
    ax.plot(distances_m, tilts, linewidth=2.5, color=color,
            label=f'{height_m} m tower')

# Comfort zones (horizontal bands)
ax.axhspan(0, COMFORTABLE_NECK_TILT_DEG, alpha=0.12, color='green',
           label=f'Comfortable (0–{COMFORTABLE_NECK_TILT_DEG:.0f}°)')
ax.axhspan(COMFORTABLE_NECK_TILT_DEG, ERGONOMIC_UPPER_BOUND_DEG, alpha=0.10, color='orange',
           label=f'Uncomfortable ({COMFORTABLE_NECK_TILT_DEG:.0f}–{ERGONOMIC_UPPER_BOUND_DEG:.0f}°)')
ax.axhspan(ERGONOMIC_UPPER_BOUND_DEG, MAX_UPWARD_HEAD_TILT_DEG, alpha=0.08, color='red',
           label=f'Strained ({ERGONOMIC_UPPER_BOUND_DEG:.0f}–{MAX_UPWARD_HEAD_TILT_DEG:.0f}°)')
ax.axhline(MAX_UPWARD_HEAD_TILT_DEG, color='darkred', linestyle='--', linewidth=1.5,
           label=f'Max head tilt ({MAX_UPWARD_HEAD_TILT_DEG:.0f}°)')

# Medieval town scale reference lines
town_refs = {
    'Town wall (~250 m)': TYPICAL_BASTIDE_DIAMETER_M / 2,
    'Bastide diameter (500 m)': TYPICAL_BASTIDE_DIAMETER_M,
    'Walking limit (1 km)': WALKING_DISTANCE_TOWN_LIMIT_M,
}
for label, dist in town_refs.items():
    ax.axvline(dist, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
    ax.text(dist + 20, 72, label, fontsize=8, color='gray', rotation=90, va='top')

ax.set_xlim(50, 2000)
ax.set_ylim(0, 80)
ax.set_xlabel('Distance to Building (m)', fontsize=12, fontweight='bold')
ax.set_ylabel('Neck Tilt to See Top (°)', fontsize=12, fontweight='bold')
ax.set_title('Is That Isekai Tower Too High?\n'
             'Neck Tilt vs Viewing Distance for Various Tower Heights',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

source_text = (
    'References:\n'
    '• Human vertical FOV ~150° (Wikipedia)\n'
    '• Medieval bastide ~500 m diameter\n'
    '• Large walled city ~0.5–1 sq mi'
)
ax.text(0.02, 0.55, source_text, transform=ax.transAxes,
        fontsize=8, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()

output_path = Path('isekai_tower_too_high.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Plot saved to: {output_path.absolute()}')

plt.show()